# NeuroVision MLflow — 7 Experiments (all models, all strategies)


## Setup: imports + helpers + MLflow tracking + callbacks


In [ ]:
import sys, os, time, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "Notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import mlflow, mlflow.tensorflow, numpy as np, tensorflow as tf
from sklearn.metrics import f1_score, classification_report
from neurovision.data import NUM_CLASSES, DEFAULT_SIZE, CLASSES, make_augmentation, make_datasets
mlflow.set_tracking_uri(f"sqlite:///{ROOT / 'mlflow.db'}")
mlflow.set_experiment("NeuroVision")
def macro_f1(model, ds):
    yt, yp = [], []
    for im, lab in ds:
        yt.append(lab.numpy())
        yp.append(np.argmax(model.predict(im, verbose=0), axis=1))
    return f1_score(np.concatenate(yt), np.concatenate(yp), average="macro")
class MacroF1(tf.keras.callbacks.Callback):
    def __init__(self, dv): super().__init__(); self.val_ds = dv
    def on_epoch_end(self, e, logs=None): (logs or {}).update({"val_f1_macro": macro_f1(self.model, self.val_ds)})
def save_curves(h, path):
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(1,2,figsize=(10,3.5))
    ax[0].plot(h.history["loss"],label="train"); ax[0].plot(h.history["val_loss"],label="val"); ax[0].set_title("Loss"); ax[0].legend()
    ax[1].plot(h.history["accuracy"],label="train"); ax[1].plot(h.history["val_accuracy"],label="val"); ax[1].set_title("Acc"); ax[1].legend()
    fig.tight_layout(); fig.savefig(path,dpi=120); plt.close(fig)
def callbacks_for(n,dv):
    out=Path(f"models/{n}"); out.mkdir(parents=True,exist_ok=True)
    return [tf.keras.callbacks.ModelCheckpoint(filepath=str(out/"best.keras"),monitor="val_loss",save_best_only=True),
            tf.keras.callbacks.BackupAndRestore(backup_dir=str(out/"backup")),
            tf.keras.callbacks.EarlyStopping(monitor="val_loss",patience=4,restore_best_weights=True),
            MacroF1(dv)]
def load_ds(prep,bs=32):
    return make_datasets("Data/raw",size=DEFAULT_SIZE,batch_size=bs,val_split=0.15,test_split=0.15,seed=42,
        augmentation=make_augmentation(0.03,0.1,0.1),preprocess=prep,cache=False).values()
def run(name,model,prep,bs,epochs,lr,dr=0.3,unfrozen="none"):
    train,val,test=load_ds(prep,bs)
    with mlflow.start_run(run_name=name) as r:
        mlflow.log_params({"model":name.split("-")[0],"strategy":name.split("-")[1],"preprocess":prep,"lr":lr,"batch_size":bs,"epochs":epochs,"dropout":dr,"img_size":224,"aug":"r+z+t","unfrozen":unfrozen,"total_params":model.count_params()})
        t0=time.perf_counter()
        h=model.fit(train,validation_data=val,epochs=epochs,callbacks=callbacks_for(name,val))
        mlflow.log_metric("train_seconds",round(time.perf_counter()-t0,1))
        mlflow.log_metric("epochs_run",len(h.history["loss"]))
        save_curves(h,f"models/{name}/curves.png"); mlflow.log_artifact(f"models/{name}/curves.png")
        tl,ta=model.evaluate(test,verbose=0)
        yt=np.concatenate([l.numpy() for _,l in test]); yp=np.argmax(model.predict(test,verbose=0),axis=1)
        mlflow.log_metrics({"test_loss":tl,"test_accuracy":ta,"test_f1_macro":f1_score(yt,yp,average="macro")})
        Path(f"models/{name}/report.txt").write_text(classification_report(yt,yp,target_names=list(CLASSES),digits=4))
        mlflow.log_artifact(f"models/{name}/report.txt")
        mlflow.tensorflow.log_model(model,artifact_path="model")
        bp=Path(f"models/{name}/best.keras")
        if bp.exists(): mlflow.log_artifact(str(bp))
        print("OK",name,"run:",r.info.run_id,f"acc={ta:.3f}")
    return r.info.run_id
print("helpers loaded")


## Data download (optional, cloud GPU)

In [ ]:
import os
from azure.storage.blob import BlobServiceClient
from pathlib import Path
conn=os.getenv("AZURE_STORAGE_CONNECTION_STRING")
if conn:
    svc=BlobServiceClient.from_connection_string(conn)
    cnt=svc.get_container_client("kcrowly-vast")
    ld=Path("Data")/"raw"
    if not ld.exists() or not any(ld.iterdir()):
        print("Download Data/raw...")
        for b in cnt.list_blobs():
            if not b.name.startswith("Data/raw/"): continue
            rel=b.name[len("Data/raw/"):]
            if not rel: continue
            p=ld/rel; p.parent.mkdir(parents=True,exist_ok=True)
            with open(p,"wb") as f: cnt.download_blob(b.name).readinto(f)
        print("done")
    else: print("Data/raw exists")
else: print("No AZURE_STORAGE_CONNECTION_STRING")

## 1. CNN scratch (scratch)

In [ ]:
train,val,test=load_ds("scale01",32)
model=tf.keras.Sequential([tf.keras.layers.Input(shape=(224,224,3)),tf.keras.layers.Conv2D(32,3,"same","relu"),tf.keras.layers.MaxPooling2D(),tf.keras.layers.Conv2D(64,3,"same","relu"),tf.keras.layers.MaxPooling2D(),tf.keras.layers.Conv2D(128,3,"same","relu"),tf.keras.layers.MaxPooling2D(),tf.keras.layers.Conv2D(256,3,"same","relu"),tf.keras.layers.GlobalAveragePooling2D(),tf.keras.layers.Dropout(0.3),tf.keras.layers.Dense(NUM_CLASSES,activation="softmax")],name="cnn_scratch")
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),loss="sparse_categorical_crossentropy",metrics=["accuracy"])
run("cnn-scratch",model,"scale01",32,30,1e-3,0.3,"none")

## 2. MobileNetV2 transfer (transfer)

In [ ]:
base=tf.keras.applications.MobileNetV2(input_shape=(224,224,3),include_top=False,weights="imagenet")
base.trainable=False
model=tf.keras.Sequential([base,tf.keras.layers.GlobalAveragePooling2D(),tf.keras.layers.Dropout(0.3),tf.keras.layers.Dense(NUM_CLASSES,activation="softmax")],name="mobilenetv2_transfer")
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),loss="sparse_categorical_crossentropy",metrics=["accuracy"])
run("mobilenetv2-transfer",model,"mobilenet",32,20,1e-3,0.3,"none")

## 3. MobileNetV2 fine-tune (fine-tune)

In [ ]:
model=tf.keras.models.load_model("models/mobilenetv2-transfer/best.keras")
base=model.layers[0];base.trainable=True
cut=next(i for i,l in enumerate(base.layers) if l.name.startswith("block_13_expand"))
for l in base.layers[:cut]:l.trainable=False
for l in base.layers[cut:]:l.trainable=True
for l in base.layers:
    if isinstance(l,tf.keras.layers.BatchNormalization):l.trainable=False
model.compile(optimizer=tf.keras.optimizers.Adam(1e-4),loss="sparse_categorical_crossentropy",metrics=["accuracy"])
run("mobilenetv2-finetune",model,"mobilenet",32,15,1e-4,0.3,"block_13_expand")

## 4. ResNet-50 transfer (transfer)

In [ ]:
train,val,test=load_ds("resnet50",32)
base=tf.keras.applications.ResNet50(input_shape=(224,224,3),include_top=False,weights="imagenet")
base.trainable=False
model=tf.keras.Sequential([base,tf.keras.layers.GlobalAveragePooling2D(),tf.keras.layers.Dropout(0.3),tf.keras.layers.Dense(NUM_CLASSES,activation="softmax")],name="resnet50_transfer")
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),loss="sparse_categorical_crossentropy",metrics=["accuracy"])
run("resnet50-transfer",model,"resnet50",32,20,1e-3,0.3,"none")

## 5. ResNet-50 fine-tune (fine-tune)

In [ ]:
train,val,test=load_ds("resnet50",32)
model=tf.keras.models.load_model("models/resnet50-transfer/best.keras")
base=model.layers[0];base.trainable=True
cut=next(i for i,l in enumerate(base.layers) if l.name.startswith("conv5_block1"))
for l in base.layers[:cut]:l.trainable=False
for l in base.layers[cut:]:l.trainable=True
for l in base.layers:
    if isinstance(l,tf.keras.layers.BatchNormalization):l.trainable=False
model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),loss="sparse_categorical_crossentropy",metrics=["accuracy"])
run("resnet50-finetune",model,"resnet50",32,15,1e-5,0.3,"conv5_block1")

## 6. VGG-16 transfer (transfer)

In [ ]:
train,val,test=load_ds("vgg16",16)
base=tf.keras.applications.VGG16(input_shape=(224,224,3),include_top=False,weights="imagenet")
base.trainable=False
model=tf.keras.Sequential([base,tf.keras.layers.GlobalAveragePooling2D(),tf.keras.layers.Dropout(0.4),tf.keras.layers.Dense(NUM_CLASSES,activation="softmax")],name="vgg16_transfer")
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),loss="sparse_categorical_crossentropy",metrics=["accuracy"])
run("vgg16-transfer",model,"vgg16",16,20,1e-3,0.4,"none")

## 7. VGG-16 fine-tune (fine-tune)

In [ ]:
train,val,test=load_ds("vgg16",16)
model=tf.keras.models.load_model("models/vgg16-transfer/best.keras")
base=model.layers[0];base.trainable=True
cut=next(i for i,l in enumerate(base.layers) if l.name.startswith("block4_conv1"))
for l in base.layers[:cut]:l.trainable=False
for l in base.layers[cut:]:l.trainable=True
for l in base.layers:
    if isinstance(l,tf.keras.layers.BatchNormalization):l.trainable=False
model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),loss="sparse_categorical_crossentropy",metrics=["accuracy"])
run("vgg16-finetune",model,"vgg16",16,15,1e-5,0.4,"block4_conv1")

## 8. Compare + champion

In [ ]:
client=mlflow.MlflowClient()
runs=mlflow.search_runs(experiment_names=["NeuroVision"],filter_string="metrics.test_f1_macro>0",order_by=["metrics.val_f1_macro DESC"])
if not runs.empty:
    best=runs.iloc[0]
    display(runs[["tags.mlflow.runName","metrics.val_f1_macro","metrics.test_f1_macro","metrics.train_seconds"]])
    ver=mlflow.register_model(f"runs:/{best['run_id']}/model","NeuroVision").version
    client.set_registered_model_alias("NeuroVision","champion",ver)
    print(f"champion v{ver} — {best['tags.mlflow.runName']}")
    champ=mlflow.tensorflow.load_model("models:/NeuroVision@champion")
    print("loaded champion")
else: print("No finished runs")

## 9. Azure upload (optional)

In [ ]:
import os
from azure.storage.blob import BlobServiceClient
conn=os.getenv("AZURE_STORAGE_CONNECTION_STRING")
if conn:
    sz=BlobServiceClient.from_connection_string(conn).get_container_client("kcrowly-vast")
    bp=Path("models/mobilenetv2-finetune/best.keras")
    if bp.exists():
        sz.upload_blob("models/mobilenetv2-finetune/best.keras",open(bp,"rb"),overwrite=True)
        print("uploaded")
    else: print("No best.keras")
else: print("No AZURE...")